# Marketing Analytics: CausalImpact & Robyn on Snowflake

An end-to-end marketing analytics workflow demonstrating how to run
R-native packages -- Google's **CausalImpact** and Meta's **Robyn** --
directly in Snowflake Workspace Notebooks, with data in Snowflake,
Feature Store pipelines, and Model Registry governance.

**What you'll do:**
1. Explore marketing data already in Snowflake
2. Build reusable Feature Store pipelines (aggregation, PIVOT, OBJECT_AGG)
3. Run CausalImpact to measure a campaign's causal effect
4. Run Robyn for Marketing Mix Modeling
5. Register models in the Snowflake Model Registry

**Database layout** (created by the setup notebook):

| Schema | Contents |
|---|---|
| `RAW_DATA` | Source tables -- campaigns, ad spend, revenue, holidays |
| `FEATURES` | Feature Store -- entity, feature views |
| `TRAINING` | Training datasets |
| `MODELS` | Model Registry -- versioned models, metrics |

**Prerequisites:** Run `workspace_marketing_setup.ipynb` first to install the
R environment, create the database/schemas, and generate source data.

**Sections:**
1. Setup and Connect
2. Exploratory Data Analysis
3. Feature Store
4. CausalImpact: Measuring Campaign Effect
5. Robyn: Marketing Mix Modeling
6. Model Registry Overview
7. Cleanup

## 1. Setup and Connect

The R environment and packages were installed by the setup notebook.
Here we configure rpy2, load packages, and connect to the data.

In [ ]:
from sfnb_setup import setup_notebook
setup_notebook(config="snowflaker_marketing_config.yaml")

In [ ]:
%%R
library(snowflakeR)
library(CausalImpact)
library(tidyverse)
library(jsonlite)
library(scales)
rcat("snowflakeR", as.character(packageVersion("snowflakeR")), "loaded")
rcat("CausalImpact", as.character(packageVersion("CausalImpact")), "loaded")

### Connect and Configure Database Context

In [ ]:
%%R
conn <- sfr_connect()

ma_db       <- "MARKETING_ANALYTICS_ML"
ma_source   <- "RAW_DATA"
ma_features <- "FEATURES"
ma_training <- "TRAINING"
ma_registry <- "MODELS"

fqn_source <- function(name) paste(ma_db, ma_source, name, sep = ".")

tbl_campaigns <- fqn_source("CAMPAIGNS")
tbl_spend     <- fqn_source("DAILY_AD_SPEND")
tbl_revenue   <- fqn_source("DAILY_MARKET_REVENUE")
tbl_holidays  <- fqn_source("HOLIDAY_CALENDAR")

n <- sfr_query(conn, paste("SELECT COUNT(*) AS n FROM", tbl_spend))
rcat(sprintf("Connected. %s ad spend records in %s.",
             format(n$N, big.mark = ","), tbl_spend))
conn

In [ ]:
%%R
reg <- sfr_model_registry(conn,
  database = ma_db,
  schema   = ma_registry
)
rcat(sprintf("Model Registry: %s.%s", ma_db, ma_registry))

## 2. Exploratory Data Analysis

Quick look at the marketing data before building models.

### Ad Spend by Channel Over Time

In [ ]:
%%R -w 900 -h 400
spend_weekly <- sfr_query(conn, paste(
  "SELECT DATE_TRUNC('week', DATE)::DATE AS WEEK_DATE,",
  "  CHANNEL, SUM(SPEND) AS TOTAL_SPEND",
  "FROM", tbl_spend,
  "GROUP BY 1, 2 ORDER BY 1, 2"
))
spend_weekly$WEEK_DATE <- as.Date(spend_weekly$WEEK_DATE)

suppressWarnings(print(
  ggplot(spend_weekly, aes(x = WEEK_DATE, y = TOTAL_SPEND, color = CHANNEL)) +
    geom_line(linewidth = 0.6) +
    scale_y_continuous(labels = comma) +
    labs(title = "Weekly Ad Spend by Channel",
         x = "Week", y = "Total Weekly Spend ($)", color = "Channel") +
    theme_minimal(base_size = 13) +
    theme(plot.title = element_text(face = "bold"), legend.position = "top")
))

### Revenue Trends by Market

The red dashed line marks the start of the Montreal TV campaign.
Look for a visible uplift in Montreal's revenue after July 2025.

In [ ]:
%%R -w 900 -h 400
revenue_daily <- sfr_query(conn, paste(
  "SELECT DATE, MARKET, REVENUE FROM", tbl_revenue,
  "ORDER BY DATE, MARKET"
))
revenue_daily$DATE <- as.Date(revenue_daily$DATE)

suppressWarnings(print(
  ggplot(revenue_daily, aes(x = DATE, y = REVENUE, color = MARKET)) +
    geom_line(alpha = 0.2) +
    geom_smooth(method = "loess", span = 0.1, se = FALSE) +
    geom_vline(xintercept = as.Date("2025-07-01"),
               linetype = "dashed", color = "red") +
    annotate("text", x = as.Date("2025-07-15"),
             y = max(revenue_daily$REVENUE) * 0.95,
             label = "Campaign Start", hjust = 0, color = "red") +
    scale_y_continuous(labels = comma) +
    labs(title = "Daily Revenue by Market",
         subtitle = "Montreal TV campaign starts July 2025",
         x = "Date", y = "Daily Revenue ($)", color = "Market") +
    theme_minimal(base_size = 13) +
    theme(plot.title = element_text(face = "bold"), legend.position = "top")
))

### Montreal vs Control Markets (Monthly)

Smoothed monthly view makes the intervention effect clearer.
Toronto and Vancouver serve as control markets (Q3 2025).

In [ ]:
%%R -w 900 -h 400
monthly_rev <- revenue_daily %>%
  mutate(month = floor_date(DATE, "month")) %>%
  group_by(month, MARKET) %>%
  summarise(avg_revenue = mean(REVENUE), .groups = "drop")

suppressWarnings(print(
  ggplot(monthly_rev, aes(x = month, y = avg_revenue, color = MARKET)) +
    geom_line(linewidth = 1) +
    geom_vline(xintercept = as.Date("2025-07-01"),
               linetype = "dashed", color = "red") +
    scale_y_continuous(labels = comma) +
    labs(title = "Monthly Average Revenue: Montreal vs Control Markets",
         subtitle = "Montreal shows uplift during Q3 2025 campaign period",
         x = "Month", y = "Average Daily Revenue ($)", color = "Market") +
    theme_minimal(base_size = 13) +
    theme(plot.title = element_text(face = "bold"), legend.position = "top")
))

## 3. Feature Store

Define reusable feature pipelines that transform raw marketing data
into model-ready formats. Each Feature View encapsulates a specific
transformation -- Snowflake handles execution and refresh.

| Feature View | Transformation | Snowflake technique | Feeds |
|---|---|---|---|
| `WEEKLY_CHANNEL_SPEND_FV` | Daily campaign spend to weekly channel totals | `GROUP BY` week + conditional aggregation | Robyn |
| `WEEKLY_REVENUE_FV` | Daily market revenue to weekly national totals | `GROUP BY` week + `SUM` | Robyn |
| `DAILY_MARKET_REVENUE_FV` | Daily revenue packed as JSON object by market | `OBJECT_AGG` | CausalImpact |

In [ ]:
%%R
fs <- sfr_feature_store(conn,
  database  = ma_db,
  schema    = ma_features,
  warehouse = conn$warehouse,
  create    = TRUE
)

dbi_con     <- sfr_dbi_connection(conn)
spend_tbl   <- tbl(dbi_con, I(tbl_spend))
revenue_tbl <- tbl(dbi_con, I(tbl_revenue))

week_entity <- sfr_create_entity(
  fs, "REPORTING_WEEK", "WEEK_DATE",
  desc = "Weekly reporting period"
)

date_entity <- sfr_create_entity(
  fs, "REPORT_DATE", "DATE",
  desc = "Daily reporting date"
)

rcat("Feature Store, dbplyr table references, and entities ready.")
fs

### Feature View 1: Weekly Channel Spend (dplyr/dbplyr pipeline)

Aggregates daily campaign-level ad spend into weekly totals per channel,
pivoted to wide format. This is the core data transformation for Robyn --
it needs one column per media variable.

We build the feature engineering query as a **dplyr/dbplyr pipeline** in R,
then render it to a **SQL string** with `dbplyr::sql_render()`. The SQL
is passed to `sfr_create_feature_view()`. This lets analysts author feature
logic in familiar R, while the pipeline executes entirely in Snowflake.

Uses a Dynamic Table (`refresh_freq = '1 hour'`) so features refresh
automatically as new spend data arrives.

In [ ]:
%%R
spend_query <- spend_tbl |>
  mutate(WEEK_DATE = as.Date(floor_date(DATE, "week"))) |>
  group_by(WEEK_DATE) |>
  summarise(
    TV_S            = sum(ifelse(CHANNEL == "TV",       SPEND, 0), na.rm = TRUE),
    FACEBOOK_S      = sum(ifelse(CHANNEL == "FACEBOOK", SPEND, 0), na.rm = TRUE),
    FACEBOOK_I      = sum(ifelse(CHANNEL == "FACEBOOK", IMPRESSIONS, 0), na.rm = TRUE),
    SEARCH_S        = sum(ifelse(CHANNEL == "SEARCH",   SPEND, 0), na.rm = TRUE),
    SEARCH_CLICKS_P = sum(ifelse(CHANNEL == "SEARCH",   CLICKS, 0), na.rm = TRUE),
    .groups = "drop"
  )

spend_sql <- dbplyr::sql_render(spend_query)
rcat("dplyr pipeline rendered to SQL:")
rcat(spend_sql)
rcat("Registered WEEKLY_CHANNEL_SPEND_FV")

In [ ]:
%%R
fv_channel_spend <- sfr_create_feature_view(
  fs,
  name         = "WEEKLY_CHANNEL_SPEND_FV",
  version      = "v1",
  entities     = week_entity,
  features     = spend_sql,
  refresh_freq = "1 hour",
  desc         = "Weekly channel spend pivot (Dynamic Table, from dplyr pipeline)",
  overwrite    = TRUE
)

rcat("Registered WEEKLY_CHANNEL_SPEND_FV (using SQL rendered from dplyr)")

### Feature View 2: Weekly Revenue (inline dplyr)

Aggregates daily market revenue into weekly national totals.
This provides the dependent variable (revenue) and conversions
for the Robyn model.

Here we pass the **dplyr/dbplyr lazy table directly** to
`sfr_create_feature_view()` -- contrast with the SQL string
approach used for FV1. Either way the pipeline runs entirely
in Snowflake.

In [ ]:
%%R
fv_weekly_revenue <- sfr_create_feature_view(
  fs,
  name     = "WEEKLY_REVENUE_FV",
  version  = "v1",
  entities = week_entity,
  features = revenue_tbl |>
    mutate(WEEK_DATE = as.Date(floor_date(DATE, "week"))) |>
    group_by(WEEK_DATE) |>
    summarise(
      REVENUE     = sum(REVENUE, na.rm = TRUE),
      CONVERSIONS = sum(CONVERSIONS, na.rm = TRUE),
      .groups = "drop"
    ),
  refresh_freq = "1 hour",
  desc      = "Weekly national revenue totals (Dynamic Table, inline dplyr)",
  overwrite = TRUE
)
rcat("Registered WEEKLY_REVENUE_FV (inline dplyr pipeline)")

### Feature View 3: Daily Market Revenue (OBJECT_AGG)

Packs per-market daily revenue into a single Snowflake `OBJECT` column
using `OBJECT_AGG`. Market names become keys in the JSON object.

This is **deliberately generic** -- the Feature View never hardcodes
market names. New markets appear as keys automatically. The R code
downstream selects which markets to unpack for a given analysis
(treatment vs control).

In [ ]:
%%R
fv_market_revenue <- sfr_create_feature_view(
  fs,
  name     = "DAILY_MARKET_REVENUE_FV",
  version  = "v1",
  entities = date_entity,
  features = paste(
    "SELECT DATE,",
    "  OBJECT_AGG(MARKET, REVENUE::VARIANT) AS MARKET_REVENUE",
    "FROM", tbl_revenue,
    "GROUP BY DATE"
  ),
  desc      = "Daily revenue as OBJECT keyed by market (OBJECT_AGG)",
  overwrite = TRUE
)
rcat("Registered DAILY_MARKET_REVENUE_FV")

In [ ]:
%%R
rcat("Feature Views registered:")
sfr_list_feature_views(fs)

### Generate Training Data for Robyn

Build a weekly date spine using dplyr (evaluated as SQL by dbplyr) and
join it with both Robyn Feature Views to produce a flat training table.
The Feature Store handles the joins on `WEEK_DATE`.

In [ ]:
%%R
week_spine <- revenue_tbl |>
  mutate(WEEK_DATE = as.Date(floor_date(DATE, "week"))) |>
  distinct(WEEK_DATE)

robyn_training <- sfr_generate_training_data(
  fs,
  spine = week_spine,
  features = list(fv_channel_spend, fv_weekly_revenue)
)

tbl_training <- paste(ma_db, ma_training, "ROBYN_TRAINING", sep = ".")
sfr_write_table(conn, tbl_training, robyn_training, overwrite = TRUE)

rcat(sprintf("Training data: %d rows x %d cols -> %s",
             nrow(robyn_training), ncol(robyn_training), tbl_training))
head(as.data.frame(robyn_training), 5)

## 4. CausalImpact: Measuring Campaign Effect

Google's CausalImpact uses Bayesian structural time series to estimate
the causal effect of an intervention. We ask: *"Did the Montreal Q3 2025 TV
campaign actually drive incremental revenue, or was it a normal
seasonal pattern?"*

The model uses Toronto and Vancouver as control markets to build a
counterfactual -- what Montreal's revenue *would have been* without
the campaign.

In [ ]:
%%R
date_spine <- revenue_tbl |>
  distinct(DATE) |>
  arrange(DATE)

market_data <- sfr_retrieve_features(
  fs,
  spine = date_spine,
  features = list(fv_market_revenue)
)

rcat(sprintf("Retrieved %d rows x %d cols from DAILY_MARKET_REVENUE_FV",
             nrow(market_data), ncol(market_data)))
head(market_data, 3)

### Unpack Market Data

The `MARKET_REVENUE` column is a Snowflake OBJECT (returned as a JSON
string). We unpack it in R using `jsonlite` and `tidyr`, selecting only
the markets needed for this analysis. This is the flexible part --
changing markets is a one-line edit, not a Feature View change.

In [ ]:
%%R
treatment <- "MONTREAL"
controls  <- c("TORONTO", "VANCOUVER")

ci_data <- market_data %>%
  mutate(DATE = as.Date(DATE)) %>%
  mutate(parsed = lapply(MARKET_REVENUE, fromJSON)) %>%
  tidyr::unnest_wider(parsed) %>%
  select(DATE, all_of(c(treatment, controls))) %>%
  arrange(DATE)

rcat(sprintf("CausalImpact data: %d days, %d series (1 treatment + %d controls)",
             nrow(ci_data), ncol(ci_data) - 1, length(controls)))
head(ci_data)

### Fit CausalImpact Model

Define the pre-intervention period (before the campaign) and the
post-intervention period (during the campaign). CausalImpact learns
the relationship between treatment and control markets in the pre-period,
then predicts the counterfactual in the post-period.

In [ ]:
%%R
intervention_date <- as.Date("2025-07-01")
pre_end    <- max(which(ci_data$DATE < intervention_date))
post_start <- min(which(ci_data$DATE >= intervention_date))

pre.period  <- c(1, pre_end)
post.period <- c(post_start, nrow(ci_data))

rcat(sprintf("Pre-period:  rows %d-%d (%s to %s)",
             pre.period[1], pre.period[2],
             ci_data$DATE[pre.period[1]], ci_data$DATE[pre.period[2]]))
rcat(sprintf("Post-period: rows %d-%d (%s to %s)",
             post.period[1], post.period[2],
             ci_data$DATE[post.period[1]], ci_data$DATE[post.period[2]]))

data_matrix <- as.matrix(ci_data[, c(treatment, controls)])
impact <- CausalImpact(data_matrix, pre.period, post.period)
rcat("CausalImpact model fitted.")

### Results

Three panels:
1. **Original** -- actual Montreal revenue vs counterfactual (predicted without campaign)
2. **Pointwise** -- day-by-day causal effect (actual minus counterfactual)
3. **Cumulative** -- running total of incremental revenue from the campaign

In [ ]:
%%R -w 900 -h 600
plot(impact)

In [ ]:
%%R
summary(impact)

In [ ]:
%%R
summary(impact, "report")

### Register CausalImpact Model in Model Registry

Register the underlying bsts (Bayesian Structural Time Series) model
in the Snowflake Model Registry. All inference dependencies (bsts, Boom,
zoo) are on conda-forge, so the model can be deployed to SPCS.

In [ ]:
%%R
mv_ci <- sfr_log_model(
  reg,
  model        = impact$model$bsts.model,
  model_name   = "CAUSAL_IMPACT_CAMPAIGN",
  predict_pkgs = c("bsts"),
  predict_body = paste(
    'nd_{{UID}} <- as.matrix({{INPUT}}[, c("TORONTO", "VANCOUVER"), drop = FALSE])',
    'pred_{{UID}} <- predict({{MODEL}}, horizon = nrow(nd_{{UID}}), newdata = nd_{{UID}})',
    'result_{{UID}} <- data.frame(',
    '  period = seq_len(nrow(nd_{{UID}})),',
    '  point_forecast = as.numeric(pred_{{UID}}$mean),',
    '  lower_95 = as.numeric(pred_{{UID}}$interval[, 1]),',
    '  upper_95 = as.numeric(pred_{{UID}}$interval[, 2])',
    ')',
    sep = '\n'
  ),
  conda_deps   = c("r-bsts", "r-boom", "r-zoo"),
  input_cols   = list(TORONTO = "double", VANCOUVER = "double"),
  output_cols  = list(
    period = "integer", point_forecast = "double",
    lower_95 = "double", upper_95 = "double"
  ),
  comment = "CausalImpact bsts model - Montreal Q3 2025 TV campaign"
)

rcat(sprintf("Registered: %s version %s",
             mv_ci$model_name, mv_ci$version_name))

## 5. Robyn: Marketing Mix Modeling

Meta's Robyn performs semi-automated Marketing Mix Modeling using ridge
regression with Adstock and saturation transformations. It uses Python's
`nevergrad` optimizer (via `reticulate`) for multi-objective hyperparameter
tuning.

In our Workspace environment, R runs inside Python via rpy2. We need to
configure `reticulate` to discover the existing Python environment rather
than creating a new virtual environment.

### Configure Python Bridge

Point `reticulate` at the notebook's Python. In a standard R environment
you would call `virtualenv_create()` -- here we skip that because rpy2
already runs R inside the notebook's Python process.

### Load Robyn and Prepare Training Data

Read the training data generated by Feature Store and rename columns
to match Robyn's expected naming convention. Also read the holiday
calendar from Snowflake.

In [ ]:
%%R
library(Robyn)

# Point cmdstanr at conda-installed cmdstan (avoids rstan recompilation)
if (requireNamespace("cmdstanr", quietly = TRUE)) {
  cmdstan_bin <- Sys.which("cmdstan")
  if (nzchar(cmdstan_bin)) {
    cmdstanr::set_cmdstan_path(dirname(dirname(cmdstan_bin)))
  }
}

robyn_data <- robyn_training %>%
  rename(DATE = WEEK_DATE) %>%
  mutate(DATE = as.Date(DATE)) %>%
  rename_with(tolower) %>%
  arrange(date)

holidays <- sfr_query(conn, paste("SELECT * FROM", tbl_holidays))
holidays <- holidays %>%
  rename_with(tolower) %>%
  mutate(ds = as.Date(ds), year = as.integer(format(ds, "%Y")))

rcat(sprintf("Robyn data: %d weeks x %d cols", nrow(robyn_data), ncol(robyn_data)))
rcat(sprintf("Holidays: %d entries", nrow(holidays)))
head(robyn_data)

### Model Specification

Define hyperparameter ranges for three channels using geometric adstock
(one decay parameter per channel). The search space is kept minimal
for demo speed.

**Robyn Bug Workaround — Hyperparameter Naming**

We discovered a bug in Robyn's `robyn_inputs()` (Use Case 1 code path in
`R/inputs.R`) that causes `robyn_run()` to fail with:

```
Error in names(hyper_list_all) <- `*vtmp*` :
  'names' attribute [1] must be the same length as the vector [0]
```

The bug is triggered when `paid_media_spends` and `paid_media_vars` use
different column names (e.g. spend = `facebook_s`, exposure = `facebook_i`).
Internally, `check_hyperparameters()` correctly renames the hyperparameters
from spend-based to var-based names — but stores the result in a **local
variable** and never writes it back to `InputCollect$hyperparameters`.
When `robyn_run()` later calls `hyper_collector()`, it looks for var-based
names that don't exist and crashes.

**Workaround:** define hyperparameters using the `paid_media_vars` names
directly (`facebook_i_*`, `search_clicks_p_*`) instead of the
`paid_media_spends` names (`facebook_s_*`, `search_s_*`). This bypasses the
broken renaming path entirely.

In [ ]:
%%R
# Hyperparameter names must use paid_media_vars names (not paid_media_spends)
# when spends and vars differ.  Robyn's robyn_inputs() has a bug where the
# internal spend-to-var renaming is not persisted to InputCollect.
hyperparameters <- list(
  tv_s_alphas = c(0.5, 3),
  tv_s_gammas = c(0.3, 1),
  tv_s_thetas = c(0.3, 0.8),
  facebook_i_alphas = c(0.5, 3),
  facebook_i_gammas = c(0.3, 1),
  facebook_i_thetas = c(0, 0.3),
  search_clicks_p_alphas = c(0.5, 3),
  search_clicks_p_gammas = c(0.3, 1),
  search_clicks_p_thetas = c(0, 0.3)
)

InputCollect <- robyn_inputs(
  dt_input = robyn_data,
  dt_holidays = holidays,
  date_var = "date",
  dep_var = "revenue",
  dep_var_type = "revenue",
  prophet_vars = c("trend", "season", "holiday"),
  prophet_country = "CA",
  paid_media_spends = c("tv_s", "facebook_s", "search_s"),
  paid_media_vars = c("tv_s", "facebook_i", "search_clicks_p"),
  window_start = "2024-01-01",
  window_end = "2026-02-28",
  adstock = "geometric",
  hyperparameters = hyperparameters
)

print(InputCollect)

### Train Model

Run the optimizer using all available cores. For a live demo we use
reduced iterations (500) and a single trial (~30 sec). Production runs
typically use 2000+ iterations and 5 trials for convergence.

In [ ]:
%%R
n_cores <- 5L
# n_cores <- max(1L, parallel::detectCores() - 1L)
rcat(sprintf("Training with %d cores", n_cores))

OutputModels <- robyn_run(
  InputCollect = InputCollect,
  cores = n_cores,
  iterations = 200,
  trials = 1,
  ts_validation = FALSE,
  add_penalty_factor = FALSE
)

### View Results

Generate Pareto-optimal model outputs. The `export = FALSE` and
`plot_pareto = FALSE` flags are set for headless notebook execution.

In [ ]:
%%R
OutputCollect <- robyn_outputs(
  InputCollect, OutputModels,
  pareto_fronts = "auto",
  csv_out = "pareto",
  clusters = TRUE,
  export = FALSE,
  plot_pareto = FALSE
)

print(OutputCollect)

### Serving Architecture Discussion

Robyn and its dependency `lares` are **not on conda-forge**, which means
the model cannot be served directly via `sfr_deploy_model()` on SPCS.
Three production-ready alternatives:

**Option A: Extract the glmnet model (recommended for scoring)**
Register only the underlying ridge regression (`glmnet`) with manually
coded Adstock/saturation transforms. `glmnet` IS on conda-forge.

**Option B: Pre-compute response curves (recommended for planning)**
Export Robyn's response curves as a Snowflake table or Python model.
A simple SQL JOIN or interpolation UDF serves spend-to-response lookups
with no R dependencies at inference.

**Option C: Model Registry for governance + custom SPCS for serving**
Register the full Robyn model in the Model Registry for versioning,
metrics, and lineage tracking. Serve from a custom SPCS container with
Robyn pre-installed. `sfr_log_model()` accepts this (the conda-forge
check is a warning, not a blocker).

## 6. Model Registry Overview

Both the CausalImpact model and any Robyn models are tracked in the
same Snowflake Model Registry -- versioned, with metrics, governed.

In [ ]:
%%R
rcat("Registered models:")
sfr_show_models(reg)

In [ ]:
%%R
rcat("CausalImpact model versions:")
sfr_show_model_versions(reg, "CAUSAL_IMPACT_CAMPAIGN")

## 7. Model Inference

The CausalImpact bsts model is now registered in the Model Registry. Below we
demonstrate two local inference paths:

1. **R** -- predict directly from the in-memory bsts model (the analyst workflow)
2. **Python** -- retrieve the model from the registry via `ModelVersion.load()` and predict (cross-language roundtrip)

A final reference section shows how to deploy the model as a production SPCS
service for R, Python, and SQL access.

In [ ]:
%%R
bsts_model <- impact$model$bsts.model

# bsts needs future values for the regression controls (TORONTO, VANCOUVER).
# Use the last 30 days of observed control data as a realistic forecast input.
newdata <- tail(data_matrix[, controls, drop = FALSE], 30)
pred <- predict(bsts_model, horizon = nrow(newdata), newdata = newdata)

forecast_r <- data.frame(
  period         = seq_len(nrow(newdata)),
  point_forecast = as.numeric(pred$mean),
  lower_95       = as.numeric(pred$interval[, 1]),
  upper_95       = as.numeric(pred$interval[, 2])
)

rcat("30-day forecast from in-memory bsts model:")
head(forecast_r, 10)

### Retrieve from Model Registry and Predict (Python)

Load the registered R model back from the Model Registry using the Python SDK.
`ModelVersion.load()` downloads the model artifacts (including the `.rds` file)
and reconstructs the `CustomModel` wrapper. Because the Workspace notebook has
rpy2, R, and bsts installed, the wrapper can invoke R prediction under the hood.

In [ ]:
from snowflake.ml.registry import Registry
from snowflake.snowpark.context import get_active_session
import rpy2.robjects as ro
import pandas as pd

session = get_active_session()
reg_py = Registry(session=session,
                  database_name="MARKETING_ANALYTICS_ML",
                  schema_name="MODELS")

m = reg_py.get_model("CAUSAL_IMPACT_CAMPAIGN")
mv = m.default

model = mv.load(force=True)

# Build input_df from the same control data used by the R cell
ctrl_r = ro.r('tail(data_matrix[, controls, drop = FALSE], 30)')
input_df = pd.DataFrame({
    "TORONTO": list(ctrl_r.rx(True, "TORONTO")),
    "VANCOUVER": list(ctrl_r.rx(True, "VANCOUVER")),
})
forecast_py = model.predict(input_df)
print("30-day forecast retrieved from Model Registry (Python):")
print(forecast_py.head(10))

### Production Serving via SPCS (Reference)

For production workloads the registered model can be deployed as an auto-scaling
REST service on Snowpark Container Services. All CausalImpact dependencies
(`bsts`, `Boom`, `zoo`) are on conda-forge, so SPCS image build succeeds.

**Deploy the service (R):**

```r
sfr_create_compute_pool(conn, "MARKETING_POOL",
                        instance_family = "CPU_X64_M")
sfr_create_image_repo(conn, "MARKETING_ANALYTICS_ML.MODELS.MARKETING_IMAGES")

sfr_deploy_model(
  reg,
  model_name   = "CAUSAL_IMPACT_CAMPAIGN",
  version_name = mv_ci$version_name,
  service_name = "causal_impact_svc",
  compute_pool = "MARKETING_POOL",
  image_repo   = "MARKETING_ANALYTICS_ML.MODELS.MARKETING_IMAGES",
  force        = TRUE
)
sfr_wait_for_service(reg, "causal_impact_svc", timeout_min = 10)
```

**Score from R:**

```r
input_df <- data.frame(horizon = rep(1L, 30))
preds <- sfr_predict(reg, "CAUSAL_IMPACT_CAMPAIGN", input_df,
                     version_name = mv_ci$version_name,
                     service_name = "causal_impact_svc")
```

**Score from Python:**

```python
result = mv.run(
    session.create_dataframe(pd.DataFrame({"HORIZON": [1] * 30})),
    service_name="causal_impact_svc"
)
result.show()
```

**Score from SQL:**

```sql
SELECT t.*, MARKETING_ANALYTICS_ML.MODELS.CAUSAL_IMPACT_SVC!PREDICT(HORIZON) AS forecast
FROM (SELECT 1 AS HORIZON FROM TABLE(GENERATOR(ROWCOUNT => 30))) t
```

The SQL path means any Snowflake user, BI tool, or scheduled task can call the
model with no R or Python required.

## 8. Cleanup

Uncomment to remove models and Feature Store resources.
Source data tables and schemas are managed by the setup notebook.

In [ ]:
# Cleanup: uncomment blocks below as needed.

# -- Delete models --
# from snowflake.snowpark.context import get_active_session
# _sess = get_active_session()
# _sess.sql("DROP MODEL IF EXISTS MARKETING_ANALYTICS_ML.MODELS.CAUSAL_IMPACT_CAMPAIGN").collect()
# print("Models deleted.")

print("Cleanup section -- uncomment blocks above to delete resources")

## Summary

This notebook demonstrated a complete marketing analytics workflow in
Snowflake with **schema-level isolation** for each concern:

| Schema | What we did |
|---|---|
| `RAW_DATA` | Queried source marketing data |
| `FEATURES` | Created 3 Feature Views (weekly spend pivot, weekly revenue, daily market OBJECT_AGG) |
| `TRAINING` | Generated Robyn training data from Feature Store joins |
| `MODELS` | Registered CausalImpact model with metrics |

### Key Capabilities Demonstrated

| Capability | How |
|---|---|
| R packages in Snowflake | CausalImpact and Robyn running natively via rpy2 |
| Feature Store | Reusable data pipelines: aggregation, PIVOT, OBJECT_AGG |
| Semi-structured data | OBJECT_AGG for flexible market-level analysis |
| Model Registry | Version control, metrics, governance for R models |
| Cross-language inference | R model retrieved from MR and scored in Python |
| SPCS serving (reference) | Production deployment pattern for R, Python, SQL access |

### Next Steps

- Connect to real marketing data sources in Snowflake
- Add more media channels to the Robyn model
- Implement Robyn response curve serving (Option A or B)
- Schedule periodic model retraining with Snowflake Tasks
- Explore model monitoring with Snowflake ML Observability

In [ ]:
# CI results -- writes a results table when run non-interactively
try:
    from snowflake.snowpark.context import get_active_session
    _ci_session = get_active_session()
    _ci_session.sql("""
        CREATE OR REPLACE TABLE NOTEBOOK_CI.MARKETING_DEMO_RESULTS AS
        SELECT 'marketing_demo' AS TEST_NAME, 'PASS' AS STATUS,
               CURRENT_TIMESTAMP() AS RUN_AT, CURRENT_USER() AS RUN_BY
    """).collect()
    print("CI results written to NOTEBOOK_CI.MARKETING_DEMO_RESULTS")
except Exception:
    pass